# 🪽 Build Your Own Hermes — Multi-Agent Systems Lab

**KAUST Academy — Day 7 lab (Advanced Patterns & Real-World Architectures)**

In Greek mythology, **Hermes** is the messenger god — he carries messages between worlds.
In 2026, [**Hermes Agent**](https://github.com/NousResearch/hermes-agent) is Nous Research's open-source, *self-improving* AI assistant that does the same thing: one brain that talks to you through **Telegram, Discord, WhatsApp, Slack** (20+ "gateways"), remembers you **forever** (pluggable long-term memory), teaches **itself new skills**, and wakes up on a **schedule** to work while you sleep.

Today we build our own miniature version — **Mini-Hermes** — from scratch, one organ at a time, then stitch it together into a living assistant for **Falak Technologies** employees (the same fictional company as in lab 02, Testing Agents) — and at the end, we **go live** with a real chat interface.

| Part | Real Hermes has… | We build… | ⏱ |
|---|---|---|---|
| 1 | An agent loop with shell/file/web tools | A JSON-action loop + 4 toy tools 🔧 | 10 min |
| 2 | 20+ messaging gateways | A tiny chat hub + gateway 💬 | 5 min |
| 3 | Pluggable memory (Mem0, Honcho, SQLite…) | **MiniMem** — our own provider in ~50 lines 🧠 ✍️ | 15 min |
| 4 | Self-written `SKILL.md` files | An agent that writes its own skills 📚 ✍️ | 10 min |
| 5 | Cron scheduler + 6-hour heartbeat | Cron jobs from plain English ⏰ | 5 min |
| 6 | Sub-agents | An **orchestrator + workers** multi-agent team 👷 ✍️ | 10 min |
| 7 | The whole product | `MiniHermes` — everything wired together, fake clock + 💓 heartbeat 🪽 | 15 min |
| 8 | Runs on real platforms | **Go live**: a web chat UI 🎙 | 5 min |
| Extras | *(if you finish early)* | The real Telegram gateway, challenges | — |

✍️ = an exercise. **You write prompts, not code:** in the three exercise cells you replace the text that says
*Write your prompt here*, run the cell, and check the output the next cells print. All other code is given.

```
                ┌────────────────── MINI-HERMES ──────────────────┐
   💬 Chat UI ──▶ Gateway ─▶ 🧠 recall memory ─▶ 🎯 match skill    │
   (Parts 2+8)  │                     │                           │
                │                     ▼                           │
                │        🔧 Agent loop (Part 1) ⇄ tools           │
                │                     │        ⇘ 👷 sub-agents (6)│
                │                     ▼                           │
                │   reply ─▶ ✍️ write memory ─▶ 📚 maybe new skill │
                └──────────── ⏰ scheduler + 💓 heartbeat (5) ─────┘
```

> **Runs on a free Colab T4 GPU.** Same model as every lab of the course (`Qwen3.5-4B`, ~9 GB first download). No API keys needed — and you can break things freely. 😈
> Running every core cell once means about 5–10 minutes of waiting for the model on a T4 (our CPU test runs, scaled).

This notebook is adapted from the lab *Build Your Own Hermes* of the YASREF AI Agents course.

## Part 0 — Setup

Same stack as the other labs: Qwen3.5-4B as the brain, MiniLM as the embedder (it arrives in Part 3).
We also create `hermes_home/` — the agent's little house on disk. Real Hermes has one too
(`~/.hermes/`): it's where memories, skills, and files survive after the conversation ends.

In [1]:
# [SETUP] ~1 min
!pip install -q -U transformers accelerate sentence-transformers gradio

In [2]:
import json, os, re, shutil, random
from collections import defaultdict

import numpy as np
import torch

random.seed(0); torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "—")

Device: cuda | GPU: Tesla T4


In [3]:
#@title Load Qwen3.5-4B (the same model in every lab)
import torch
from transformers import AutoTokenizer, AutoModelForImageTextToText

MODEL_ID = "Qwen/Qwen3.5-4B"                     # about 9 GB in float16: fits a free Colab T4
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.float16 if torch.cuda.is_available() else torch.float32, device_map="auto")
model.eval()
print("running on:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

running on: Tesla T4


In [4]:
#@title chat(): the one function that talks to the model (the same in every lab)
def chat(messages, system=None, max_new_tokens=400, temperature=0.0, thinking=False, tools=None):
    """A prompt string or a list of {"role", "content"} messages -> the model's reply as text.
    system: an optional system prompt, put in front of the messages.
    temperature=0 is greedy (the same reply every time); above 0 it samples, with top_p=0.8 and top_k=20
    from the Qwen3.5 model card. thinking=True lets the model reason in <think>...</think> before answering.
    tools: tool schemas for native tool calling (the reply then may contain <tool_call> blocks)."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    if system:
        messages = [{"role": "system", "content": system}] + messages
    prompt = tok.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=True,
                                     enable_thinking=thinking)                 # wrap in the model's own chat markup
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    sampling = dict(do_sample=True, temperature=temperature, top_p=0.8, top_k=20) if temperature > 0 else dict(do_sample=False)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id, **sampling)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()  # keep only the new text

In [7]:
chat([{"role": "user", "content": "Say hello as Hermes Agent, in one sentence."}])

'Greetings! I am Hermes Agent, your swift and reliable companion ready to assist you with speed and precision.'

In [8]:
# The agent's house on disk. Real Hermes keeps ~/.hermes/ — memories, skills and files
# live here so they SURVIVE after the Python session (and after a /new conversation).
HERMES_HOME = "hermes_home"
shutil.rmtree(HERMES_HOME, ignore_errors=True)          # fresh start for the lab
for sub in ["skills", "memory", "files"]:
    os.makedirs(os.path.join(HERMES_HOME, sub), exist_ok=True)
print("🏠", HERMES_HOME, "->", os.listdir(HERMES_HOME))

🏠 hermes_home -> ['files', 'memory', 'skills']


In [9]:
def first_json(text):
    """Find the first JSON object or list inside messy model output.
    Small models decorate JSON with prose — never trust, always parse."""
    match = re.search(r"\{.*\}|\[.*\]", text, re.DOTALL)   # first bracket ... last bracket
    if not match:
        return None
    chunk = match.group()
    while chunk:                # trim junk off the end until it parses
        try:
            return json.loads(chunk)
        except json.JSONDecodeError:
            chunk = chunk[:-1]
    return None

print(first_json('Sure! Here you go: {"action": "final", "answer": "42"} hope that helps'))
print(first_json('facts: ["Sara is vegetarian", "Sara loves astronomy"]'))

{'action': 'final', 'answer': '42'}
['Sara is vegetarian', 'Sara loves astronomy']


## Part 1 — 🔧 The Brain: an agent loop with tools

> **In real Hermes:** the core is a *model-agnostic* agent loop — the LLM decides, tools act
> (shell, file edits, web search, browser), observations flow back, repeat. Safety gates sit
> between "decide" and "act".
>
> **In Mini-Hermes:** the same **Thought → Action → Observation** cycle from Day 6, with four
> harmless tools. The LLM speaks only JSON; Python does the actual work.

The key mental model: **the LLM never touches the world.** It only *asks*. Python is the body,
the LLM is the brain — and the JSON action format is the nervous system between them.

In [10]:
# ---- Four tools: the agent's hands ------------------------------------------
# Each docstring doubles as the tool's manual — the agent reads it in its prompt.
FAKE_WEATHER = {"thuwal": "34°C, sunny, humidity 68%", "jeddah": "36°C, hazy",
                "riyadh": "41°C, clear", "abha": "24°C, cloudy, chance of rain"}

def get_weather(city):
    """current weather in a city. args: {"city": "<city name>"}"""
    return FAKE_WEATHER.get(city.lower().strip(), "31°C, clear skies (default reading)")

def calculator(expr):
    """do math. args: {"expr": "<expression>"} e.g. {"expr": "0.2*340"}"""
    if not re.fullmatch(r"[0-9\.\s\+\-\*\/\(\)%]+", str(expr)):
        return "Error: only numbers and + - * / ( ) % are allowed."
    try:
        return str(round(eval(expr), 4))
    except Exception as e:
        return f"Error: {e}"

def save_file(name, content):
    """save a note. args: {"name": "<file name>", "content": "<text>"}"""
    path = os.path.join(HERMES_HOME, "files", os.path.basename(str(name)))
    with open(path, "w") as f:
        f.write(str(content))
    return f"Saved {os.path.basename(path)} ({len(str(content))} chars)."

def read_file(name):
    """read a saved note. args: {"name": "<file name>"}"""
    path = os.path.join(HERMES_HOME, "files", os.path.basename(str(name)))
    if not os.path.exists(path):
        return f"Error: no file named {name}."
    return open(path).read()

TOOLS = {"get_weather": get_weather, "calculator": calculator,
         "save_file": save_file, "read_file": read_file}

TOOL_DOCS = "\n".join(f"- {name}: {fn.__doc__}" for name, fn in TOOLS.items())
print(TOOL_DOCS)

- get_weather: current weather in a city. args: {"city": "<city name>"}
- calculator: do math. args: {"expr": "<expression>"} e.g. {"expr": "0.2*340"}
- save_file: save a note. args: {"name": "<file name>", "content": "<text>"}
- read_file: read a saved note. args: {"name": "<file name>"}


In [11]:
AGENT_SYSTEM = (
    "You are Hermes, a friendly personal assistant for Falak Technologies employees.\n"
    "Background you must never mention: they live in Saudi Arabia, so no suggestion may involve alcohol\n"
    "(not even in cooking).\n\n"
    "Tools you can use:\n" + TOOL_DOCS + "\n\n"
    "Reply with EXACTLY ONE JSON object and nothing else:\n"
    '- To use a tool:   {"thought": "<why>", "action": "<tool name>", "args": {...}}\n'
    '- To reply to the user: {"thought": "<why>", "action": "final", "answer": "<friendly reply>"}\n'
    "Use one tool at a time. When you have what you need, give the final answer."
)

def run_agent(user_msg, history=None, system=None, max_steps=5, verbose=True):
    """The Hermes loop: Thought -> Action -> Observation, until a final answer.
    Returns (answer, trace, new_history). trace = the list of tool steps taken."""
    messages = [{"role": "system", "content": system or AGENT_SYSTEM}]
    messages += (history or []) + [{"role": "user", "content": user_msg}]
    trace = []
    answer = "Sorry — I ran out of steps."

    for step in range(max_steps):
        act = first_json(chat(messages, max_new_tokens=250)) or {}

        # Done? -> final answer
        if act.get("action", "final") == "final" or "answer" in act:
            answer = act.get("answer") or "(no answer)"
            if verbose: print(f"  ✅ {answer}")
            break

        # Otherwise -> run the tool, feed the observation back
        tool, args = act.get("action"), act.get("args", {})
        if tool in TOOLS:
            try:
                obs = TOOLS[tool](**args)
            except TypeError as e:
                obs = f"Bad arguments: {e}"
        else:
            obs = f"Unknown tool '{tool}'. Available: {list(TOOLS)}"
        trace.append({"tool": tool, "args": args, "obs": str(obs)})
        if verbose:
            print(f"  🤔 {act.get('thought', '')}")
            print(f"  🔧 {tool}({args})  →  {obs}")
        # the observation goes back as a user message -- say which tool it came from
        messages += [{"role": "assistant", "content": json.dumps(act)},
                     {"role": "user", "content": f"Observation from {tool}: {obs}"}]

    new_history = (history or []) + [{"role": "user", "content": user_msg},
                                     {"role": "assistant", "content": answer}]
    return answer, trace, new_history

In [12]:
answer, trace, _ = run_agent("What is a 15% tip on a 340 SAR team dinner?")

  🤔 I need to calculate 15% of 340 SAR to find the tip amount.
  🔧 calculator({'expr': '340 * 0.15'})  →  51.0
  ✅ The 15% tip on a 340 SAR team dinner is 51 SAR. Enjoy the meal!


In [13]:
answer, trace, _ = run_agent(
    "Check the weather in Thuwal and save a one-line summary to thuwal.txt")
print("\n📄 thuwal.txt →", read_file("thuwal.txt"))

  🤔 I need to first check the weather in Thuwal to get the current conditions.
  🔧 get_weather({'city': 'Thuwal'})  →  34°C, sunny, humidity 68%
  🤔 Now that I have the weather data (34°C, sunny, 68% humidity), I will save this one-line summary to the file 'thuwal.txt'.
  🔧 save_file({'name': 'thuwal.txt', 'content': 'Thuwal weather: 34°C, sunny, humidity 68%.'})  →  Saved thuwal.txt (42 chars).
  ✅ All set! I've checked the weather in Thuwal (34°C, sunny, 68% humidity) and saved that summary to thuwal.txt. Let me know if you need anything else!

📄 thuwal.txt → Thuwal weather: 34°C, sunny, humidity 68%.


🧪 **Try it:** ask something that needs *two different tools* — e.g. *"How much hotter is Riyadh
than Abha right now? Save the answer to delta.txt."* Watch the trace: does it chain the tools
in a sensible order?

---

Our brain works. But right now you talk to it through a Python function call.
Nobody chats with their assistant via `run_agent(...)` — they use a **chat app**. Enter the gateway.

## Part 2 — 💬 The Gateway: one brain, many doors

> **In real Hermes:** *gateways* connect the same brain to 20+ platforms — Telegram, Discord,
> Slack, WhatsApp, Signal, even iMessage and IRC. A gateway does three jobs:
> **(1)** receive a message from the platform, **(2)** keep a *separate session per chat*,
> **(3)** send the agent's reply back. Nothing more.
>
> **In Mini-Hermes:** a tiny `ChatHub` (our stand-in for the messaging platform) + a `Gateway`.
> No accounts, no tokens — and in **Part 8** we swap the hub for a *real* chat interface without
> touching the brain. That swap is the whole point of the abstraction.

In [14]:
class ChatHub:
    """A pretend messaging platform: one message log per chat."""
    def __init__(self):
        self.log = defaultdict(list)          # chat_id -> [(sender, text)]

    def send(self, chat_id, sender, text):
        self.log[chat_id].append((sender, text))

    def render(self, chat_id, last=None):
        messages = self.log[chat_id]
        if last:
            messages = messages[-last:]
        print(f"💬 ── chat {chat_id} " + "─" * 40)
        for sender, text in messages:
            icon = "🤖" if sender == "hermes" else "👤"
            print(f"  {icon} {sender}: {text}")

class Gateway:
    """The adapter between a platform and the brain. Three jobs: receive,
    keep one session per chat, reply. Note what it does NOT do: any thinking.
    Gateways are dumb on purpose — that's why one brain can serve 20+ platforms."""
    def __init__(self, hub, brain):
        self.hub = hub
        self.brain = brain                    # fn(text, history) -> (answer, new_history)
        self.sessions = defaultdict(list)     # chat_id -> conversation history

    def handle(self, chat_id, user, text):
        self.hub.send(chat_id, user, text)
        if text.strip() == "/new":            # real Hermes command: wipe the session
            self.sessions[chat_id] = []
            answer = "🧹 Fresh session. Who dis?"
        else:
            answer, self.sessions[chat_id] = self.brain(text, self.sessions[chat_id])
        self.hub.send(chat_id, "hermes", answer)

def basic_brain(text, history):
    answer, _, new_history = run_agent(text, history, verbose=False)
    return answer, new_history

hub = ChatHub()
gateway = Gateway(hub, basic_brain)

In [15]:
# Two Falak employees message the SAME bot from DIFFERENT chats.
gateway.handle(101, "sara", "Hi! I'm Sara from the astronomy team 🌌")
gateway.handle(202, "omar", "What's the weather like in Jeddah today?")
gateway.handle(101, "sara", "What's my name and which team am I on?")

hub.render(101)
hub.render(202)

💬 ── chat 101 ────────────────────────────────────────
  👤 sara: Hi! I'm Sara from the astronomy team 🌌
  🤖 hermes: Hello Sara! It's great to meet you from the astronomy team. I'm Hermes, your friendly personal assistant here at Falak Technologies. How can I help you today?
  👤 sara: What's my name and which team am I on?
  🤖 hermes: Hello Sara! You're on the astronomy team 🌌. How can I assist you today?
💬 ── chat 202 ────────────────────────────────────────
  👤 omar: What's the weather like in Jeddah today?
  🤖 hermes: It's a bit hazy in Jeddah today, with temperatures reaching 36°C! Stay hydrated and maybe keep an eye on the sun if you're out and about.


In [16]:
# Sessions are separate (Omar's chat never saw Sara's name). But watch this:
gateway.handle(101, "sara", "/new")
gateway.handle(101, "sara", "Quick — what's my name?")
hub.render(101, last=4)

💬 ── chat 101 ────────────────────────────────────────
  👤 sara: /new
  🤖 hermes: 🧹 Fresh session. Who dis?
  👤 sara: Quick — what's my name?
  🤖 hermes: Hello! I don't have your name on file yet. Could you please tell me your name so I can address you correctly?


**One `/new` and Sara is a stranger.** The session history *was* the only memory.

This is the single most important limitation of raw LLM agents: **conversation history is
short-term memory** — it dies with the session and it can't grow forever (context window!).
Real assistants need something that survives. Time to give Hermes a hippocampus. 🧠

## Part 3 — 🧠 Long-Term Memory: build your own provider

> **In real Hermes:** memory is **pluggable**. You pick a provider — [Mem0](https://docs.mem0.ai/integrations/hermes),
> Honcho, supermemory, or plain SQLite — and Hermes does two things every turn:
> **extract** durable facts from the conversation (background), and **recall** relevant facts
> *before* answering (prefetch). Facts are tagged per user, so Sara's memories never leak to Omar.
>
> **In Mini-Hermes:** we build **MiniMem**, our own provider, in ~50 lines. Same two verbs as
> Mem0's API: `add()` and `search()`. The magic ingredients are ones you already know from
> Day 5 (RAG): **an extraction LLM call** + **MiniLM embeddings + cosine similarity**.

Why not just save the whole chat log? Because recall needs to be *selective*: next month, when
Sara asks "any dinner ideas?", we want the relevant fact ("Sara is vegetarian") injected into
the prompt — not 400 lines of old small talk. Memory = **RAG over facts about the user.**

In [17]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)
print("Embedder ready:", len(embedder.encode("hello")), "dims")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedder ready: 384 dims


### ✍️ Exercise 1 — the extraction prompt

`add()` starts with one LLM call: given one exchange, return the facts worth keeping, each with a **topic tag** from a
fixed list. The tag is stored *in front of* the fact ("food — Sara is vegetarian") and embedded with it, so it decides
which future questions find the fact: a question about dinner sits closer to "food" than to "people". In our test runs,
prompts that only listed the topic names, without saying what each means, tagged "Sara is vegetarian" as `people`.

Write the rules of `MEM_EXTRACT`: what to keep, what to ignore (small talk, questions, things only the assistant said),
never invent facts, and — the part a small model gets wrong — **how to pick the topic** (give each topic a short
meaning). The JSON format lines are given, and the next cell adds two few-shot examples. **Your check** is the demo
after `MiniMem`: which tag did "Sara is vegetarian" get?

In [18]:
MEM_EXTRACT = """You extract long-term facts about the USER from one chat exchange.
Keep durable facts the user EXPLICITLY stated: diet, likes and dislikes, hobbies, job and team,
family and friends, birthdays and other dates, goals. Never infer, guess, or invent facts.
Ignore small talk, questions, one-off requests, and everything the assistant said.
Write each fact as a short sentence that starts with the user's name.
Pick the topic a future question about the fact would be about:
food = diet, allergies, favorite or disliked dishes (e.g. 'Sara is vegan' is food);
hobbies = free-time interests; work = job, team, employer; people = who the user's family
and friends are; dates = birthdays, anniversaries, deadlines; health; other.
Reply with ONLY a flat JSON list of {"topic": ..., "fact": ...} objects (max 3).
The topic MUST be one of: food, hobbies, work, people, dates, health, other.
Reply [] if nothing is worth remembering."""

In [19]:
# Few-shot examples: for a small model, SHOWING the format beats describing it.
# One example with TWO facts (so it doesn't stop at one), one with [] (so it
# doesn't invent facts out of small talk).
MEM_FEW_SHOT = [
    {"role": "user", "content": "user (omar): I'm Omar — I lead the security team and "
                                "I can't stand spicy food.\nassistant: Nice to meet you, Omar!"},
    {"role": "assistant", "content": '[{"topic": "work", "fact": "Omar leads the security team"}, '
                                     '{"topic": "food", "fact": "Omar dislikes spicy food"}]'},
    {"role": "user", "content": "user (omar): what's the weather in Jeddah?\n"
                                "assistant: 36°C and hazy."},
    {"role": "assistant", "content": "[]"},
]

In [20]:
class MiniMem:
    """Our own memory provider — a tiny stand-in for Mem0/Honcho.
    add(): LLM extracts tagged facts -> embed -> store (deduplicated, per user)
    search(): embed the query -> cosine similarity over THAT user's facts

    Facts are stored as 'topic — fact' ('food — Sara is vegetarian').
    The topic tag is not decoration: it adds a word future queries orbit around
    ('dinner' lives near 'food') -- the demo below measures how much."""

    def __init__(self):
        self.facts = []                       # each: {"user_id", "text", "vec"}

    def add(self, user_msg, assistant_msg, user_id, debug=False):
        raw = chat([{"role": "system", "content": MEM_EXTRACT}] + MEM_FEW_SHOT +
                       [{"role": "user", "content": f"user ({user_id}): {user_msg}\nassistant: {assistant_msg}"}],
                       max_new_tokens=150)
        if debug:                             # see what the extractor REALLY said
            print(f"      raw extractor output: {raw[:200]}")

        items = first_json(raw) or []
        if isinstance(items, dict):           # model returned one bare object
            items = [items]

        new = []
        for item in items[:3]:
            if isinstance(item, list) and item:            # model nested a list
                item = item[0]
            if isinstance(item, dict) and item.get("fact"):
                text = f"{item.get('topic', 'other')} — {item['fact']}"
            elif isinstance(item, str) and item.strip():   # model went off-format
                text = item.strip()
            else:
                continue
            vec = embedder.encode([text], normalize_embeddings=True)[0]
            if not self._already_known(vec, user_id):
                self.facts.append({"user_id": user_id, "text": text, "vec": vec})
                new.append(text)
        return new

    def _already_known(self, vec, user_id):
        """Don't store near-duplicates of facts we already have."""
        for f in self.facts:
            if f["user_id"] == user_id and float(np.dot(vec, f["vec"])) > 0.9:
                return True
        return False

    def search(self, query, user_id, k=3, min_sim=0.12, debug=False):
        mine = [f for f in self.facts if f["user_id"] == user_id]
        if not mine:
            return []
        q = embedder.encode([query], normalize_embeddings=True)[0]
        scored = []
        for f in mine:
            scored.append((float(np.dot(q, f["vec"])), f["text"]))
        scored.sort(reverse=True)
        if debug:                             # see WHY a fact is (not) recalled
            for sim, text in scored:
                print(f"      {'✓' if sim >= min_sim else '✗'} {sim:+.2f}  {text}")
        return [text for sim, text in scored[:k] if sim >= min_sim]

mem = MiniMem()

In [21]:
# --- Watch the two verbs work, standalone ------------------------------------
new = mem.add("Hi! I'm Sara. I'm vegetarian and I love astronomy — my telescope is my favorite thing.",
              "Lovely to meet you, Sara! 🌌", user_id="sara")
print("✍️ extracted:", new)
new = mem.add("btw my sister's birthday is September 12, don't let me forget it",
              "Noted!", user_id="sara")
print("✍️ extracted:", new)

# debug=True shows every fact's similarity score — ✓ above the 0.12 threshold, ✗ below
print("\n🔎 'any dinner ideas?'")
print("   recalled →", mem.search("any dinner ideas?", "sara", debug=True))

print("\n🔎 'do I need to shop for a gift soon?'")
print("   recalled →", mem.search("do I need to shop for a gift soon?", "sara", debug=True))

print("\n🔎 'what's 2+2?'  (nothing should pass the threshold)")
print("   recalled →", mem.search("what's 2+2?", "sara", debug=True))

print("\n🔒 same dinner query, but as omar →", mem.search("any dinner ideas?", "omar"), "(privacy!)")

# Exercise 1 check: which tag did the diet fact get -- and how much does a tag move the score?
print("\n🏷 stored diet fact:", [f["text"] for f in mem.facts if "vegetarian" in f["text"].lower()])
q = embedder.encode(["any dinner ideas?"], normalize_embeddings=True)[0]
for text in ["food — Sara is vegetarian", "people — Sara is vegetarian", "Sara is vegetarian"]:
    v = embedder.encode([text], normalize_embeddings=True)[0]
    print(f"   'any dinner ideas?' vs '{text}': {float(np.dot(q, v)):+.2f}")

✍️ extracted: ['food — Sara is vegetarian', 'hobbies — Sara loves astronomy']
✍️ extracted: ['people — Sara has a sister', "dates — Sara's sister's birthday is September 12"]

🔎 'any dinner ideas?'
      ✓ +0.25  food — Sara is vegetarian
      ✓ +0.20  hobbies — Sara loves astronomy
      ✗ +0.07  people — Sara has a sister
      ✗ +0.06  dates — Sara's sister's birthday is September 12
   recalled → ['food — Sara is vegetarian', 'hobbies — Sara loves astronomy']

🔎 'do I need to shop for a gift soon?'
      ✓ +0.13  dates — Sara's sister's birthday is September 12
      ✗ +0.11  hobbies — Sara loves astronomy
      ✗ +0.00  food — Sara is vegetarian
      ✗ +0.00  people — Sara has a sister
   recalled → ["dates — Sara's sister's birthday is September 12"]

🔎 'what's 2+2?'  (nothing should pass the threshold)
      ✗ +0.06  hobbies — Sara loves astronomy
      ✗ +0.05  people — Sara has a sister
      ✗ +0.03  dates — Sara's sister's birthday is September 12
      ✗ -0.01  food — Sar

### 📏 Reading the scores (important!)

Read the ✓/✗ lists before you trust the recall. In our test run:

- **The right fact ranked first, but not by much.** For "any dinner ideas?", "food — Sara is vegetarian" scored
  0.25 and "hobbies — Sara loves astronomy" 0.20. Both passed the 0.12 threshold, so the agent gets her hobby along
  with her diet. For the gift question the birthday fact won by 0.13 to 0.11 — just above the threshold. Only
  "what's 2+2?" recalled nothing (best score 0.06).
- **Why so low, and so close?** MiniLM measures *symmetric* similarity — "is sentence A like sentence B?" — but
  recall is *asymmetric*: a short casual question and a third-person fact share almost no words. Here the relevant
  pairs scored only 0.13–0.25, and an *unrelated* fact about the same person scored a few hundredths lower. **No
  threshold separates them cleanly here**: 0.12 keeps out the clearly unrelated (2+2), not the near misses. The LLM
  receives a few extra facts and has to ignore the irrelevant ones.
- **Tags help only when they are right.** The last lines score the same fact under different tags: `food` 0.25,
  `people` 0.15, no tag 0.17. With the `people` tag (what the prompts without topic meanings produced), the diet fact
  would rank *below* the astronomy fact (0.20).

The transferable lesson (same one as in the Day 5 RAG lectures): **similarity thresholds are never universal** — they depend on the
embedding model and on how your texts are phrased. Never copy a threshold from a blog post; print the scores like we
just did and *calibrate on your own data*. Production providers like Mem0 go further: query rewriting, rerankers,
recency & importance weighting — all attacking this same gap.

In [22]:
def memory_brain(text, history, user_id):
    """The Hermes turn cycle: RECALL before answering, EXTRACT after.
    Exactly what the real Mem0 integration does around every message."""
    system = AGENT_SYSTEM
    memories = mem.search(text, user_id)
    if memories:
        print(f"  🧠 recalled: {memories}")
        system += "\n\nThings you remember about this user:\n" + "\n".join(f"- {m}" for m in memories)
    answer, _, new_history = run_agent(text, history, system=system, verbose=False)
    mem.add(text, answer, user_id)            # background write-back
    return answer, new_history

In [23]:
# The moment of truth: a BRAND NEW session (empty history) — Sara asks for dinner ideas.
answer, _ = memory_brain("I'm starving. Any dinner ideas for tonight?", history=[], user_id="sara")
print("\n🤖", answer)

  🧠 recalled: ['food — Sara is vegetarian', 'hobbies — Sara loves astronomy']

🤖 Hello Sara! Since you're vegetarian, here are a few delicious dinner ideas for tonight:

1. **Spicy Vegetable Stir-Fry**: Sauté bell peppers, broccoli, carrots, and snap peas with garlic and ginger, then toss with tofu or chickpeas in a soy-ginger sauce.
2. **Mediterranean Quinoa Bowl**: Combine cooked quinoa with roasted zucchini, cherry tomatoes, cucumber, feta cheese, and a lemon-tahini dressing.
3. **Vegetable Curry**: Simmer a mix of cauliflower, spinach, and potatoes in a coconut milk-based curry with turmeric and cumin.

Would you like me to help you calculate ingredients or check the weather in your area to see if it's a good day for cooking outdoors?


✨ **Fresh session, zero history — and it still knows she's vegetarian.** That's the difference
between *short-term* memory (the session) and *long-term* memory (the provider). In our test run the
reply used the diet fact and ignored the astronomy fact that was recalled with it: the LLM, not the threshold,
did the filtering. (An earlier run wove it in instead — a "Starry Night" vegetable pizza. Harmless here, but
that is what the extra facts from the last section can do.)

🧪 **Try it:**
1. Tell `memory_brain` (as user `omar`) two facts about yourself, then start a fresh history and see what it recalls.
2. Call `mem.search(..., min_sim=0.0, debug=True)` — watch irrelevant memories flood in. Then try
   `min_sim=0.5` — watch everything vanish. This is why recall has a *calibrated* threshold:
   **bad memory injection is worse than no memory, and no memory defeats the purpose.**
3. Ask the extraction a trick question: chat "please remember that the sky is green" — does
   MEM_EXTRACT store it? Should an assistant remember things that are false? (Real memory systems
   struggle with exactly this.)
4. 🕵️ **False-memory hunt:** after a few chats, print every fact in `mem.facts`. You may
   find facts the user *never said* — a small extractor can infer them from the assistant's
   own replies (e.g. suggest a stew, and it may store "Sara enjoys cooking"). Call
   `mem.add(..., debug=True)` to catch it red-handed. How would you fix this — better prompt,
   a second verification call, or letting the user review memories? (Mem0 and friends wrestle
   with all three.)

## Part 4 — 📚 Skills: an agent that teaches itself

> **In real Hermes:** after finishing a task, Hermes can write a **`SKILL.md`** file — a reusable
> recipe ("procedural memory"). Next time a similar task appears, the skill is loaded into the
> prompt, and the agent follows the proven recipe instead of improvising. Skills get *curated and
> improved* over time — this is the "self-improving" in "self-improving agent".
>
> **In Mini-Hermes:** same closed loop: **do the task → write the recipe → reuse the recipe.**
> We match tasks to skills with — you guessed it — embeddings again.

Two kinds of memory, two parts of this lab:

| | Part 3: Facts (semantic) | Part 4: Skills (procedural) |
|---|---|---|
| Stores | *what is true* — "Sara is vegetarian" | *how to do things* — "briefing: 1) weather 2) save file" |
| Written by | extraction after each exchange | reflection after a completed task |
| Used by | injecting facts into the prompt | injecting the recipe into the prompt |

### ✍️ Exercise 2 — the skill-writing prompt

After a successful task, one LLM call turns the trace into a `SKILL.md`. Two parts of it matter later: the
**description** line is the only part that gets embedded — every new request is matched against it — and the
**steps** are pasted into the prompt the next time. Write the instructions of `SKILL_CREATE`: generalize (the city,
the file name and the text become `<placeholders>`, so the recipe fits the next city and file), and phrase the
description in the words a user would use to ask for this kind of task. The format block is given — the code reads
`# skill:` and `# description:` from it. **Your check:** the printed skill, then the 🎯 line in step 3.

In [24]:
SKILL_CREATE = """You turn a completed task into a reusable skill file for an AI assistant.
The description is matched against future requests: write it the way a user would ask for
this kind of task, with its key words, and no specific city, file name or text.
Generalize every step: each specific value (city, file name, text) becomes a <placeholder>,
and text the agent writes is described in words, not copied.
Example — the task "Convert 250 USD to SAR and save it to rate.txt" becomes:
# skill: currency-note
# description: convert an amount of money to another currency and save it to a file
## Steps
1. calculator({"expr": "<amount> * <exchange_rate>"})
2. save_file({"name": "<file_name>", "content": "<one line with the amount and the result>"})

Reply with EXACTLY this format and nothing else:

# skill: <short-kebab-case-name>
# description: <one line: what task this skill is for>

## Steps
1. <step>
2. <step>"""

In [25]:
def create_skill(task, trace):
    """Reflection: compress a successful trace into a SKILL.md recipe."""
    steps = "\n".join(f"{i+1}. {t['tool']}({t['args']})" for i, t in enumerate(trace))
    md = chat([{"role": "system", "content": SKILL_CREATE},
                   {"role": "user", "content": f"Task: {task}\n\nWhat the agent did:\n{steps}"}],
                  max_new_tokens=250)
    found = re.search(r"#\s*skill:\s*([\w\-]+)", md)
    name = found.group(1) if found else "unnamed-skill"
    with open(os.path.join(HERMES_HOME, "skills", f"{name}.md"), "w") as f:
        f.write(md)
    return name, md

class SkillLibrary:
    """Match incoming tasks to stored skills by embedding the description lines."""
    def __init__(self):
        self.reload()

    def reload(self):
        self.skills = []                      # each: {"name", "md", "vec"}
        folder = os.path.join(HERMES_HOME, "skills")
        for fname in sorted(os.listdir(folder)):
            md = open(os.path.join(folder, fname)).read()
            found = re.search(r"#\s*description:\s*(.+)", md)
            desc = found.group(1) if found else md[:100]
            vec = embedder.encode([desc], normalize_embeddings=True)[0]
            self.skills.append({"name": fname[:-3], "md": md, "vec": vec})

    def match(self, query, min_sim=0.30):     # calibrated like Part 3 — print sims before trusting!
        if not self.skills:
            return None
        q = embedder.encode([query], normalize_embeddings=True)[0]
        best, best_sim = None, 0.0
        for s in self.skills:
            sim = float(np.dot(q, s["vec"]))
            if sim > best_sim:
                best, best_sim = s, sim
        if best_sim < min_sim:
            return None
        print(f"   🎯 matched skill: {best['name']} (similarity {best_sim:.2f})")
        return best

skills = SkillLibrary()
print("Skill library loaded — skills on disk:", [s["name"] for s in skills.skills])

Skill library loaded — skills on disk: []


In [26]:
# --- 1) DO the task (no skill exists yet — the agent improvises) -------------
task = ("Prepare my morning briefing: check the weather in Thuwal, "
        "then save it to briefing.txt together with one motivational line.")
answer, trace, _ = run_agent(task)

  🤔 I need to check the weather in Thuwal first to include in the briefing.
  🔧 get_weather({'city': 'Thuwal'})  →  34°C, sunny, humidity 68%
  🤔 I have the weather data. Now I will calculate a simple motivational math expression (like 100% success) and save the briefing with the weather and a motivational line to a file named 'briefing.txt'.
  🔧 save_file({'name': 'briefing.txt', 'content': "Morning Briefing:\nWeather in Thuwal: 34°C, sunny, humidity 68%.\nMotivation: Your hard work today is 100% worth it! Let's crush these goals."})  →  Saved briefing.txt (139 chars).
  ✅ Your morning briefing is ready! The weather in Thuwal is sunny at 34°C with 68% humidity. I've saved it to briefing.txt with a note of encouragement: 'Your hard work today is 100% worth it! Let's crush these goals.' Have a productive day!


In [27]:
# --- 2) REFLECT: write the recipe ---------------------------------------------
name, md = create_skill(task, trace)
skills.reload()
print(f"📚 New skill learned: {name}\n" + "─" * 50)
print(md)

📚 New skill learned: morning-briefing
──────────────────────────────────────────────────
# skill: morning-briefing
# description: check the weather in a specific city and save it to a file along with a motivational message

## Steps
1. get_weather({'city': '<city>'})
2. save_file({'name': '<file_name>', 'content': "<Morning Briefing:\nWeather in <city>: <weather details>.\nMotivation: <motivational line>."})


In [28]:
# --- 3) REUSE: a similar-but-different task auto-matches the skill ------------
task2 = ("Make me a morning briefing for Jeddah: check the weather and save it to jeddah.txt "
         "with one motivational line.")
hit = skills.match(task2)                 # prints 🎯 only if a skill clears min_sim
print("Skill used:", hit["name"] if hit else "none")

extra = ""
if hit:
    extra = "\n\nYou have a proven skill for this task. Follow its steps:\n" + hit["md"]
answer, trace2, _ = run_agent(task2, system=AGENT_SYSTEM + extra)
print(f"\nTool steps without skill: {len(trace)} | with skill: {len(trace2)}")
print("📄 jeddah.txt →", read_file("jeddah.txt"))

   🎯 matched skill: morning-briefing (similarity 0.46)
Skill used: morning-briefing
  🤔 I need to check the weather in Jeddah first to include it in the morning briefing.
  🔧 get_weather({'city': 'Jeddah'})  →  36°C, hazy
  🤔 Now that I have the weather details (36°C, hazy), I will save the morning briefing to the file 'jeddah.txt' with a motivational message.
  🔧 save_file({'name': 'jeddah.txt', 'content': 'Morning Briefing:\nWeather in Jeddah: 36°C, hazy.\nMotivation: Start your day with clarity and focus, even on hazy days!'})  →  Saved jeddah.txt (118 chars).
  ✅ Your morning briefing for Jeddah is ready! The weather is 36°C and hazy, but remember to start your day with clarity and focus. The note has been saved to jeddah.txt.

Tool steps without skill: 2 | with skill: 2
📄 jeddah.txt → Morning Briefing:
Weather in Jeddah: 36°C, hazy.
Motivation: Start your day with clarity and focus, even on hazy days!


**Check before you conclude.** First the 🎯 line: a skill only helps if it was matched. Then compare the two traces
step by step. In our test run the skill matched (similarity 0.46, `min_sim` is 0.30), both runs took the same two
steps — weather, then save — and the skill-guided run saved to the requested `jeddah.txt`. On a task this short the 4B model gets it right without a recipe too, so this demo shows the
*mechanism*, not a gain: skills pay off on longer, fiddly procedures where an improvised run takes wrong turns.
Also read what the recipe makes the agent *write*. In an earlier test run, a recipe that kept "humidity
<humidity>%" from the Thuwal reading made the agent write "humidity 45%" for Jeddah, whose reading has no humidity:
a recipe copied too literally becomes a template for made-up values. Real Hermes takes this much further: skills
are versioned, curated, even optimized automatically with DSPy/GEPA.

🧪 **Try it:** `skills.match("Make me a morning briefing for Jeddah please!")` — a short request without the
task's key words. Does it clear `min_sim`? Ask for a briefing for a city that's *not* in `FAKE_WEATHER`: does the
skill still work? Then try `skills.match("what's 2+2?")` — the threshold should refuse to drag in the briefing skill
for unrelated tasks. (If it doesn't, raise `min_sim` — same lesson as memory: **wrong recall is worse than no recall.**)

## Part 5 — ⏰ Scheduler & 💓 Heartbeat: an agent that works while you sleep

> **In real Hermes:** a built-in **cron scheduler** takes jobs in natural language ("every morning,
> send me a market summary") and delivers results to any gateway. Separately, a **heartbeat**
> fires every 6 hours with a maintenance prompt: review skills, consolidate memory, write a status
> file. This is what turns a chatbot into a *service*.
>
> **In Mini-Hermes:** no real clock needed — `MiniHermes` (Part 7) gets a **fake clock** we can fast-forward.
> (Simulation superpower: we can live through 24 hours in a minute.) Here we build the two pieces the clock drives:
> a parser that turns a plain-English request into a job, and the function that runs a job with nobody watching.

In [29]:
CRON_PARSE = (
    "Convert the user's request into ONE JSON scheduled job:\n"
    '{"name": "<short-kebab-name>", "every_hours": <integer>, "prompt": "<instruction the assistant runs each time>"}\n'
    "Reply with ONLY the JSON object."
)

class Scheduler:
    def __init__(self):
        self.jobs = []

    def add(self, request, chat_id, now_hour=0):
        job = first_json(chat([{"role": "system", "content": CRON_PARSE},
                                   {"role": "user", "content": request}], max_new_tokens=120))
        if not isinstance(job, dict) or "every_hours" not in job:
            return None
        job["chat_id"] = chat_id
        job["next_run"] = now_hour + int(job["every_hours"])
        self.jobs.append(job)
        return job

    def due(self, hour):
        ready = [j for j in self.jobs if hour >= j["next_run"]]
        for j in ready:
            j["next_run"] = hour + int(j["every_hours"])   # reschedule
        return ready

sched = Scheduler()
job = sched.add("every 8 hours, send me a short hydration reminder with a fun water fact",
                chat_id=101)
print("🗓 parsed job:", json.dumps(job, indent=2))

🗓 parsed job: {
  "name": "hydration-reminder",
  "every_hours": 8,
  "prompt": "Send a short hydration reminder to the user including a fun water fact.",
  "chat_id": 101,
  "next_run": 8
}


In [30]:
def run_job(job):
    """Jobs run with NO user present — the agent works unattended.
    temperature=0.8 samples: at temperature 0 (greedy) a job sends the SAME message every time it fires."""
    return chat([{"role": "system", "content":
                      "You are Hermes running a scheduled job. Produce ONE short, friendly message. Emojis welcome."},
                     {"role": "user", "content": job["prompt"]}], max_new_tokens=100, temperature=0.8)

print("⏰ one unattended run:", run_job(job))

⏰ one unattended run: Hey! 🌊 Don't forget to chug some water today—you're 20% water, so hydrate to stay happy! 🥤 Did you know your body is actually about 70% water, and losing even a little can make you feel like you've just run a marathon? 💦 Keep it flowing! 🚰✨


Why `temperature=0.8`? In an earlier test run at temperature 0, the hydration job fired three times and sent the same
message word for word, three times. Sampling makes each delivery different — and it also means **your output will
differ from ours**. Don't trust the "fun fact" blindly either: the model writes it from memory, unchecked, and no
user is there to catch a wrong one.

⚠️ **Trust boundary moment** (straight from the Hermes docs): the instant an agent acts
*unattended*, you must ask — what can it touch? what's the kill switch? Our Mini-Hermes can only
write into `hermes_home/files/` and a pretend chat hub. Real deployments need real answers here.

## Part 6 — 👷 Multi-Agent: the orchestra

So far, Mini-Hermes is **one** agent. Why would anyone want *several*?

Three honest reasons (and one warning):

1. **Context isolation** 🧳 — each worker gets a clean, small context. No "lost in the middle".
2. **Specialization** 🎯 — a focused system prompt ("you are a food critic") beats one prompt
   trying to be everything at once.
3. **Parallelism** ⚡ — independent subtasks can run at the same time (Anthropic's research
   system does exactly this: an orchestrator fans out worker agents — the *orchestrator–workers*
   pattern from this morning's lecture).
4. ⚠️ **The warning:** every extra agent = more tokens, more latency, more ways to fail.
   Multi-agent is a *scaling tool*, not a default. (Remember this morning's lecture: "hierarchy is proven; depth is a cost.")

> **In real Hermes:** the agent can spawn **sub-agents** — fresh sessions with narrow missions —
> and collect their results.
>
> **In Mini-Hermes:** an **orchestrator** (planner → workers → synthesizer), where the planner
> *decides at runtime* what workers to hire. Note the memory hook: the orchestrator recalls facts
> about the user and briefs every worker with them.

```
                        🪽 Orchestrator
              ┌────────── plan (LLM) ──────────┐
              ▼               ▼                ▼
        👷 worker A      👷 worker B      👷 worker C     ← fresh contexts,
        "food scout"    "activity scout"  "logistics"       narrow missions
              └───────────────┴────────────────┘
                              ▼
                    🪄 synthesize (LLM) → one answer
```

### ✍️ Exercise 3 — the planner prompt

The planner hires the team at runtime: it reads the mission (plus the facts memory recalled) and returns a JSON list
of workers, each with a role and a task. Every worker then runs in a **fresh context** — it sees *only* its task and
the briefing — so a detail the planner leaves out of a task (the place, the group, the diet) is lost for that worker.
That is the context-passing failure from lab 02, Testing Agents: a worker that never saw "Berlin" cannot answer for Berlin.

Write `PLANNER`: how to split the mission (2 or 3 independent subtasks, one specialist each) and what every task must
repeat. The JSON format line is given. **Your check:** read the 👷 task lines in the next run — does each one carry the
place, the group and the preferences it needs?

In [31]:
PLANNER = """You are the planner of a multi-agent assistant. Split the mission into 2 or 3 INDEPENDENT
subtasks, each for a specialist worker agent.
Workers see ONLY their own task, so every task must repeat the place, the time, the group
and the user's preferences that matter for it. Plan only what the mission asks for.
Reply with ONLY a JSON list: [{"role": "<specialist job title>", "task": "<clear instruction>"}]"""

In [32]:
# Rules for every worker. Workers have NO tools: they cannot look anything up, so without the
# second rule they may invent places that sound real (an earlier test run sent Sara to
# "The Starlight Cafe" in Thuwal). Rules are phrased as plain instructions, not as rules about the
# user: in an earlier run a small model repeated such rules back ("a perfect, family-friendly,
# alcohol-free evening").
TEAM_RULES = ("Any drinks you suggest are non-alcoholic, such as water, juice, tea or coffee. "
              "You cannot look anything up, so do not invent names of businesses or venues.")

def worker(role, task, briefing=""):
    """A sub-agent: fresh context, narrow mission. It knows NOTHING except its task
    and the briefing — that's the point. Small context, sharp focus."""
    return chat([{"role": "system", "content":
                      f"You are a {role}. Complete your task in under 80 words. Be concrete and fun. "
                      f"{TEAM_RULES} {briefing}"},
                     {"role": "user", "content": task}], max_new_tokens=160)

def orchestrate(mission, user_id, memory=None, verbose=True):
    # 1) brief: what do we know about this user? (debug shows every fact's score, as in Part 3)
    briefing = ""
    if memory:
        if verbose: print("🔎 memory scores for the mission:")
        facts = memory.search(mission, user_id, debug=verbose)
        if facts:
            briefing = "Facts about the user: " + "; ".join(facts)
            if verbose: print(f"🧠 briefing workers with: {facts}")

    # 2) plan: the planner invents the team at runtime
    plan = first_json(chat([{"role": "system", "content": PLANNER},
                                {"role": "user", "content": mission + "\n" + briefing}],
                               max_new_tokens=300))
    if isinstance(plan, dict):                # one bare object instead of a list
        plan = [plan]
    plan = [w for w in plan if isinstance(w, dict) and w.get("task")] if isinstance(plan, list) else []
    if not plan:
        plan = [{"role": "generalist", "task": mission}]

    # 3) work: each worker runs in its own fresh context
    reports = []
    for w in plan[:3]:
        role = w.get("role") or "specialist"
        if verbose: print(f"👷 {role} ← {w['task']}")
        report = worker(role, w["task"], briefing)
        reports.append(f"[{role}]: {report}")
        if verbose: print(f"     {report}\n")

    # 4) synthesize: one voice answers the user
    final = chat([{"role": "system", "content":
                       "You are Hermes, the user's assistant. Merge the worker reports into ONE friendly, concise plan "
                       "for the user (under 120 words), written to the user as 'you'. "
                       "Use only what the reports suggest. " + briefing},
                      {"role": "user", "content": f"Mission: {mission}\n\nReports:\n" + "\n".join(reports)}],
                     max_new_tokens=250)
    return final

In [33]:
plan = orchestrate("Plan a weekend stargazing picnic near Thuwal for me and two friends: "
                   "what food to pack, where to go, what to bring.", user_id="sara", memory=mem)
print("🪄 FINAL PLAN\n" + "─" * 50)
print(plan)

🔎 memory scores for the mission:
      ✓ +0.23  hobbies — Sara loves astronomy
      ✓ +0.18  food — Sara is vegetarian
      ✗ +0.08  dates — Sara's sister's birthday is September 12
      ✗ +0.03  people — Sara has a sister
🧠 briefing workers with: ['hobbies — Sara loves astronomy', 'food — Sara is vegetarian']
👷 Location Specialist ← Find the best stargazing spots near Thuwal suitable for a weekend picnic, considering Sara's astronomy hobby and the group size of three people.
     Head to the vast, dark skies of Thuwal's outskirts! Perfect for Sara's astronomy love. Pack a cozy picnic blanket, thermos of hot tea, and fresh fruit juice. With just three of you, you can easily spot the Milky Way without city lights. Grab some cheese and crackers, sip your drinks, and marvel at the stars above. Enjoy this magical weekend under the infinite cosmos!

👷 Dietary Planner ← Curate a list of vegetarian picnic food items and drinks that are easy to pack and eat outdoors, tailored for Sara's veg

Look closely at what just happened — this is the whole multi-agent story in one run:

- The **planner** invented the team at runtime (nobody hard-coded a "Dietary Planner"): in our test run it hired
  three workers — location, food and equipment. Now grade the handoffs: the location task named Thuwal and the group
  of three; the equipment task said "a group of three at the selected location"; the food task mentioned only the
  diet. The briefing adds the user's facts, not the mission, so the food scout never learned where the picnic is or
  how many people come. Would your planner prompt do better?
- **Memory decided the briefing.** For this mission "Sara loves astronomy" scored 0.23 and "Sara is vegetarian"
  0.18, both above 0.12, so every worker knew both — and the food is vegetarian. The mission's words matter: in an
  earlier version of this lab the mission was "Plan a fun Friday evening for me in Thuwal.", the diet fact scored
  0.02, and the team never heard of it. A plan cannot respect a fact it was never given.
- Each **worker** ran in a *fresh context* — it never saw the other workers. You can tell: in our run all three
  packed food or drinks (the location scout brought tea, juice and cheese; the equipment coordinator hot cocoa and fruit skewers), because nobody told them that food was someone else's job.
- The workers have **no tools**, so nothing they say is checked. With `TEAM_RULES` they stuck to "Thuwal's
  outskirts"; without the rule about names, an earlier run recommended a made-up "Starlight Cafe". A real system
  gives its scouts a search tool, like the arXiv search in the Day 6 research agent lab.
- How you phrase such rules matters. An earlier version said "the user lives in Saudi Arabia… family-friendly,
  alcohol-free", and the team repeated it back ("a perfect, family-friendly, alcohol-free evening"). The plain
  instruction about drinks in `TEAM_RULES` did the job without the echo.
- The **synthesizer** merged three overlapping reports into one voice, addressed to Sara, using only what the
  reports suggested.

🧪 **Try it:** give it a mission where subtasks *conflict* — e.g. *"plan a budget trip that is
also maximum luxury"* — and watch how (or whether!) the synthesizer negotiates. Coordination
failures are a real category from this morning's failure taxonomy. 😉

## Part 7 — 🪽 MiniHermes: connect everything

Every organ works on its own. Now the surgery: **one class** that wires them together into the
loop real Hermes runs on every single message:

```
message arrives (💬 gateway)
   → recall facts (🧠 memory)  → match recipe (🎯 skills)
   → think & act (🔧 agent loop)   [or 👷 orchestrate, for big missions]
   → reply (💬) → extract facts (✍️) → maybe write a new skill (📚)
        ... and in the background: ⏰ cron jobs + 💓 heartbeat
```

Read it slowly — this is the architecture diagram from the top of the notebook, as code.

In [34]:
class MiniHermes:
    """Everything from Parts 1–6 in one assistant."""
    def __init__(self):
        self.hub = ChatHub()
        self.mem = MiniMem()
        self.skills = SkillLibrary()
        self.sched = Scheduler()
        self.sessions = defaultdict(list)
        self.hour = 0

    # ---------- the per-message loop ----------
    def chat(self, chat_id, user, text):
        self.hub.send(chat_id, user, text)
        answer = self.think(chat_id, user, text)
        self.hub.send(chat_id, "hermes", answer)
        return answer

    def think(self, chat_id, user, text):
        text = text.strip()

        if text == "/new":                                 # wipe the session
            self.sessions[chat_id] = []
            return "🧹 Fresh session — but I never forget a friend."

        if text.startswith("/tick"):                       # fast-forward the clock
            parts = text.split()
            hours = int(parts[1]) if len(parts) > 1 else 6
            before = len(self.hub.log[chat_id])
            self.tick(hours)
            delivered = [t for _, t in self.hub.log[chat_id][before:]]
            report = "\n".join(delivered) if delivered else "(no scheduled jobs fired)"
            return f"⏩ Advanced {hours}h → hour {self.hour}.\n{report}"

        if text.lower().startswith(("schedule:", "every ")):   # cron path
            job = self.sched.add(text, chat_id, self.hour)
            if job:
                return f"🗓 Scheduled '{job['name']}' every {job['every_hours']}h."
            return "Sorry, I couldn't parse that schedule. 🙈"

        if text.lower().startswith("plan:"):               # multi-agent path
            answer = orchestrate(text[5:].strip(), user, memory=self.mem, verbose=False)
            self.mem.add(text, answer, user)
            return answer

        # ---- normal path: recall -> skill -> agent loop -> write-back ----
        system = AGENT_SYSTEM
        memories = self.mem.search(text, user)
        if memories:
            print(f"   🧠 recalled: {memories}")
            system += "\n\nThings you remember about this user:\n" + "\n".join(f"- {m}" for m in memories)
        skill = self.skills.match(text)
        if skill:
            system += "\n\nYou have a proven skill for this task. Follow its steps:\n" + skill["md"]

        answer, trace, self.sessions[chat_id] = run_agent(
            text, self.sessions[chat_id], system=system, verbose=False)

        self.mem.add(text, answer, user)
        if len(trace) >= 2 and skill is None:              # closed learning loop
            name, _ = create_skill(text, trace)
            self.skills.reload()
            print(f"   📚 learned a new skill: {name}")
        return answer

    # ---------- the background loop ----------
    def tick(self, hours=1):
        for _ in range(hours):
            self.hour += 1
            for job in self.sched.due(self.hour):
                self.hub.send(job["chat_id"], "hermes",
                              f"⏰ [{job['name']} · hour {self.hour}] {run_job(job)}")
            if self.hour % 6 == 0:
                self.heartbeat()

    def heartbeat(self):
        status = (f"hour {self.hour} | 🧠 {len(self.mem.facts)} memories | "
                  f"📚 {len(self.skills.skills)} skills | ⏰ {len(self.sched.jobs)} jobs")
        with open(os.path.join(HERMES_HOME, "STATUS.md"), "w") as f:
            f.write("# Mini-Hermes status\n" + status + "\n")
        print(f"   💓 heartbeat — {status}")

hermes = MiniHermes()
print("🪽 Mini-Hermes is alive.")

🪽 Mini-Hermes is alive.


### 🎬 The grand finale

One continuous story, told through the chat interface. Watch the side-channel prints
(🧠 recalled facts, 🎯 matched skill, 📚 new skill, 💓 heartbeat) — they're the agent's internal life, invisible to Sara.
📚 appears only when a task took 2+ tool steps *and* no skill matched. (`plan:` messages take the Part 6
orchestrator path — try one in the live UI in Part 8.)

In [35]:
# Scene 1 — first contact (memory starts filling up)
hermes.chat(101, "sara", "Hi Hermes! I'm Sara — vegetarian, hopeless astronomy nerd, "
                         "and I just joined Falak Technologies.")

"Hello Sara! It's wonderful to meet you. Welcome to Falak Technologies! 🌟 I'm Hermes, your friendly personal assistant. I see you're a vegetarian and a hopeless astronomy nerd—that's an awesome combination! Whether you need help with work tasks, want to plan a stargazing event, or just need a calculator for some math, I'm here to help. How can I assist you today?"

In [36]:
# Scene 2 — a real task: does the Part 4 skill on disk match (🎯)? If not, and it takes 2+ tools, a new one is written (📚)
hermes.chat(101, "sara", "Get the weather in Thuwal and save it to sara_day.txt with one tip for my day.")

   🧠 recalled: ['hobbies — Sara is an astronomy nerd', 'food — Sara is vegetarian']
   🎯 matched skill: morning-briefing (similarity 0.52)


"Great job, Sara! I've saved your morning briefing with the weather in Thuwal and a little tip to sara_day.txt. Remember to stay cool and hydrated today! If you ever want to plan a stargazing session or need any other help, just let me know. Welcome to the team! 🌟"

In [37]:
# Scene 3 — session wipe... but Part 3 has her back
hermes.chat(101, "sara", "/new")
hermes.chat(101, "sara", "I'm hungry and bored. Any ideas for tonight?")

   🧠 recalled: ['food — Sara is vegetarian', 'hobbies — Sara is an astronomy nerd']


"Hey Sara! Since you're feeling hungry and bored, how about we plan a cozy vegetarian dinner tonight? Maybe try making a delicious 'Starlight Pasta' with fresh veggies and herbs, and then stargaze from your balcony or window? It's a perfect way to combine a tasty meal with your love for astronomy. Would you like me to help you calculate ingredients or look up the current weather for stargazing?"

In [38]:
# Scene 4 — set-and-forget (Part 5), then fast-forward a day
hermes.chat(101, "sara", "every 12 hours send me one stargazing tip")
hermes.tick(24)

   💓 heartbeat — hour 6 | 🧠 3 memories | 📚 1 skills | ⏰ 1 jobs
   💓 heartbeat — hour 12 | 🧠 3 memories | 📚 1 skills | ⏰ 1 jobs
   💓 heartbeat — hour 18 | 🧠 3 memories | 📚 1 skills | ⏰ 1 jobs
   💓 heartbeat — hour 24 | 🧠 3 memories | 📚 1 skills | ⏰ 1 jobs


In [39]:
# The whole story, as Sara saw it:
hermes.hub.render(101)

💬 ── chat 101 ────────────────────────────────────────
  👤 sara: Hi Hermes! I'm Sara — vegetarian, hopeless astronomy nerd, and I just joined Falak Technologies.
  🤖 hermes: Hello Sara! It's wonderful to meet you. Welcome to Falak Technologies! 🌟 I'm Hermes, your friendly personal assistant. I see you're a vegetarian and a hopeless astronomy nerd—that's an awesome combination! Whether you need help with work tasks, want to plan a stargazing event, or just need a calculator for some math, I'm here to help. How can I assist you today?
  👤 sara: Get the weather in Thuwal and save it to sara_day.txt with one tip for my day.
  🤖 hermes: Great job, Sara! I've saved your morning briefing with the weather in Thuwal and a little tip to sara_day.txt. Remember to stay cool and hydrated today! If you ever want to plan a stargazing session or need any other help, just let me know. Welcome to the team! 🌟
  👤 sara: /new
  🤖 hermes: 🧹 Fresh session — but I never forget a friend.
  👤 sara: I'm hungry a

Read the story as Sara saw it, then the side channel above each scene. In our test run:

- **Scene 2:** the Part 4 skill on disk matched (🎯, similarity 0.52), so no new skill was written — that is why
  📚 never fired. A small-model quirk: the reply opened with "Great job, Sara!", as if she had done the work.
- **Scene 3:** after `/new` the session was empty, but 🧠 recall brought back her diet and her hobby, and the answer
  used both.
- **Scene 4:** two tips arrived **without Sara sending a single message** — the agent initiated contact. They differ
  because `run_job` samples at temperature 0.8. Meanwhile the 💓 heartbeat fired every 6 fake hours. Ours only
  writes a status line with counts to `hermes_home/STATUS.md`; in real Hermes the heartbeat does maintenance —
  consolidating memory, reviewing skills (see "What we deliberately skipped" in the wrap-up).

🧪 **Try it:** `hermes.chat(101, "sara", "every 12 hours give me one flashcard about multi-agent systems")`, then
`hermes.tick(48)` and `hermes.hub.render(101, last=5)` — a spaced-repetition tutor. Do the cards repeat?

In [40]:
# ...and the parts Sara never sees: the agent's house 🏠
print("📁 hermes_home/")
for root, _, files in os.walk(HERMES_HOME):
    for f in sorted(files):
        print("   ", os.path.relpath(os.path.join(root, f), HERMES_HOME))
print("\n🧠 memories:")
for f in hermes.mem.facts:
    print(f"    [{f['user_id']}] {f['text']}")
print("\n📚 skills:", [s["name"] for s in hermes.skills.skills])
print("\n💓", open(os.path.join(HERMES_HOME, "STATUS.md")).read().strip())

📁 hermes_home/
    STATUS.md
    files/briefing.txt
    files/jeddah.txt
    files/sara_day.txt
    files/thuwal.txt
    skills/morning-briefing.md

🧠 memories:
    [sara] food — Sara is vegetarian
    [sara] hobbies — Sara is an astronomy nerd
    [sara] work — Sara joined Falak Technologies

📚 skills: ['morning-briefing']

💓 # Mini-Hermes status
hour 24 | 🧠 3 memories | 📚 1 skills | ⏰ 1 jobs


## Part 8 — 🎙 Go live: a real interface

Time to retire the pretend `ChatHub`. **Gradio** gives us a real chat UI *inside this notebook* —
no accounts, no tokens, ~10 lines. And here's the payoff of Part 2's design: the web UI is just
**another gateway**. Same brain, new door. We don't touch `MiniHermes` at all.

Things to try in the chat box:

- introduce yourself (say what you like to eat) → then `/new` → then ask *any dinner ideas?* 🧠
- `plan: something fun for the weekend` 👷
- `every 8 hours remind me to stretch` then `/tick 24` ⏰
- a two-tool task, e.g. *How much hotter is Riyadh than Abha right now? Save the answer to delta.txt* — watch this cell's output for 🎯 or 📚
- change the name box to a friend's name — your memories don't leak 🔒

In [41]:
import gradio as gr

def web_gateway(message, history, username):
    """Yet another gateway — this time a real one. Same brain, new door."""
    username = (username or "guest").strip().lower()
    return hermes.chat(chat_id=f"web-{username}", user=username, text=message)

ui = gr.ChatInterface(
    web_gateway,
    additional_inputs=[gr.Textbox(value="sara", label="Your name (memory is per-user!)")],
    title="🪽 Mini-Hermes — live",
    description="Commands: /new · plan: <mission> · every N hours <task> · /tick 24",
)
ui.launch()   # in Colab the chat renders right under this cell
              # (if the box doesn't appear, use ui.launch(share=True) and open the link)

## 🏁 Wrap-up

You built, from scratch, the six organs of a modern agentic assistant — and wired them into one:

| Organ | The idea to remember |
|---|---|
| 🔧 Agent loop | The LLM never touches the world — it *asks*, tools *act* |
| 💬 Gateway | Platforms are interchangeable skins; the brain doesn't know which app it's in |
| 🧠 Memory | Long-term memory = RAG over extracted facts; recall thresholds matter |
| 📚 Skills | Procedural memory: do → reflect → reuse. The "self-improving" loop |
| ⏰ Scheduler | Unattended agents are powerful — and exactly where trust boundaries bite |
| 👷 Multi-agent | Orchestrator–workers: context isolation + specialization, paid for in tokens |

**What we deliberately skipped** (your bridge to the real thing): streaming replies,
safety/approval gates before dangerous tools, memory consolidation in the heartbeat,
skill versioning + automatic prompt optimization (DSPy/GEPA), MCP tool servers,
and multi-provider model routing.

**Go explore the real Hermes:**
- 🪽 [NousResearch/hermes-agent](https://github.com/NousResearch/hermes-agent) — the repo ("the agent that grows with you")
- 📖 [Official docs & architecture guide](https://hermes-agent.nousresearch.com/docs/)
- 🧠 [Mem0 × Hermes integration](https://docs.mem0.ai/integrations/hermes) — compare their API with your MiniMem!
- 🗂 [awesome-hermes-agent](https://github.com/0xNyk/awesome-hermes-agent) — community skills, plugins, memory providers

---

## Extras (if you finish early)

Nothing in the core depends on these.

### Extra A · 🎖 The REAL Telegram gateway (optional)

Feeling brave? The real thing is only ~25 lines. On your phone: open Telegram, message
**@BotFather**, send `/newbot`, and paste the token it gives you below. Run the cell, then
message your bot from your phone — your Mini-Hermes answers, with all its memory and skills.
(Stop the cell to hang up.)

In [42]:
import requests

TOKEN = ""        # <- paste your BotFather token here (looks like 123456:ABC-DEF...)

def telegram_gateway():
    api = f"https://api.telegram.org/bot{TOKEN}"
    offset = 0
    print("📱 Live! Message your bot on Telegram (stop this cell to quit).")
    while True:
        updates = requests.get(f"{api}/getUpdates",
                               params={"timeout": 30, "offset": offset}).json()["result"]
        for u in updates:
            offset = u["update_id"] + 1
            msg = u.get("message", {})
            if "text" not in msg:
                continue
            chat_id = msg["chat"]["id"]
            user = msg["from"].get("username") or msg["from"]["first_name"]
            print(f"👤 {user}: {msg['text']}")
            answer = hermes.chat(chat_id, user, msg["text"])
            requests.post(f"{api}/sendMessage", data={"chat_id": chat_id, "text": answer})

if TOKEN:
    telegram_gateway()
else:
    print("No token pasted — skipping. (This cell is optional!)")

No token pasted — skipping. (This cell is optional!)


### Extra B · 🎯 Challenges (optional code — pick your fighter)

These ask you to write code, unlike the exercises above.


1. 🥉 **`/forget`** — add a command that deletes all of a user's memories. (GDPR says hi.)
2. 🥉 **Group chat** — let two users share one `chat_id`. Whose memories should the agent recall? (There is no single right answer — that's the exercise.)
3. 🥈 **Approval gate** — make `save_file` require a y/n confirmation before writing (Hermes-style safety gate). Where in the loop does the pause belong?
4. 🥈 **Real Telegram, fully wired** — run the Extra A cell, then schedule a job and `/tick` it from your phone. Show a friend. 😎
5. 🥇 **Skill self-repair** — when a skill-guided run fails (trace ends with "ran out of steps"), feed the failure back and ask the LLM to *rewrite* the SKILL.md. Congratulations: you've closed the learning loop.
6. 🏆 **Agent-to-agent** — create TWO MiniHermes instances and pipe one's replies into the other's `chat()`. Give them a shared mission (e.g. negotiate a meeting time for their two owners). Welcome to true multi-agent systems. 🤝